In [3]:
pip install torch

79.17s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 MB 44.9 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pinocchio as pin
import torch
import time

# ----------------------------
# GPU + dtype setup
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64  # match numpy double precision
print("Using device:", device)

# ----------------------------
# Algorithm Implementations
# ----------------------------
def gauss_jordan(M, b):
    return torch.linalg.solve(M, b)

def neumann_series_inverse(M, num_terms=10):
    M0 = torch.diag(torch.diag(M))
    M0_inv = torch.linalg.inv(M0)
    E = torch.eye(M.shape[0], device=M.device, dtype=M.dtype) - M0_inv @ M
    S = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    term = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    for _ in range(1, num_terms):
        term = term @ E
        S = S + term
    return S @ M0_inv

def spai_inverse(M):
    return torch.diag(1.0 / torch.diag(M))

def hala(M, b, num_neumann=5):
    G_spai = spai_inverse(M)
    E = torch.eye(M.shape[0], device=M.device, dtype=M.dtype) - G_spai @ M
    S = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    term = torch.eye(M.shape[0], device=M.device, dtype=M.dtype)
    for _ in range(1, num_neumann):
        term = term @ E
        S = S + term
    M_inv_hala = S @ G_spai
    qddot_hala = M_inv_hala @ b
    error = torch.linalg.norm(M @ qddot_hala - b)
    if error > 1e-3:
        qddot_hala = gauss_jordan(M, b)
    return qddot_hala

# ----------------------------
# Setup Pinocchio model
# ----------------------------
urdf_path = '/Users/user_1/mlpro/HALA/ur5robot.urdf'
model = pin.buildModelFromUrdf(urdf_path)
data = model.createData()
nq, nv = model.nq, model.nv

q_lower, q_upper = model.lowerPositionLimit, model.upperPositionLimit
qd_limit = model.velocityLimit

# Ensure finite, reasonable velocity limits
qd_limit_safe = qd_limit.copy()
qd_limit_safe[~torch.isfinite(torch.tensor(qd_limit_safe))] = 1.0
qd_limit_safe = torch.clamp(torch.tensor(qd_limit_safe), 0, 10).numpy()

# ----------------------------
# Random Sampling
# ----------------------------
num_runs = 1000
q_lower_t = torch.tensor(q_lower, dtype=dtype)
q_upper_t = torch.tensor(q_upper, dtype=dtype)
qd_limit_safe_t = torch.tensor(qd_limit_safe, dtype=dtype)

# Uniformly sample between lower and upper joint limits
q_samples = q_lower_t + (q_upper_t - q_lower_t) * torch.rand(num_runs, nq, dtype=dtype)
qd_samples = -qd_limit_safe_t + 2 * qd_limit_safe_t * torch.rand(num_runs, nv, dtype=dtype)

# ----------------------------
# Benchmark Setup
# ----------------------------
results = {
    'ref_time': [], 'neumann_time': [], 'spai_time': [], 'hala_time': [],
    'neumann_error': [], 'spai_error': [], 'hala_error': [],
    'ref_kappa': [], 'neumann_kappa': [], 'spai_kappa': [], 'hala_kappa': []
}

# ----------------------------
# Benchmark Loop
# ----------------------------
for i in range(num_runs):
    q = q_samples[i].cpu().numpy()
    qd = qd_samples[i].cpu().numpy()
    pin.computeAllTerms(model, data, q, qd)

    # Convert Pinocchio outputs to GPU tensors
    M = torch.tensor(data.M, dtype=dtype, device=device)
    Cqd = torch.tensor(data.nle - data.g, dtype=dtype, device=device)
    g_vec = torch.tensor(data.g, dtype=dtype, device=device)
    tau = torch.ones(nv, dtype=dtype, device=device)
    b = tau - Cqd - g_vec

    # Perturbation for stability
    delta_M = torch.randn_like(M) * 1e-6
    M_pert = M + delta_M

    # --- Reference (Gauss-Jordan) ---
    torch.cuda.synchronize()
    t0 = time.time()
    qddot_ref = gauss_jordan(M, b)
    torch.cuda.synchronize()
    t1 = time.time()
    results['ref_time'].append((t1 - t0) * 1000)

    qddot_ref_pert = gauss_jordan(M_pert, b)
    kappa_ref = (torch.linalg.norm(qddot_ref_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['ref_kappa'].append(kappa_ref.item())

    # --- Neumann Series ---
    torch.cuda.synchronize()
    t0 = time.time()
    neumann_inv = neumann_series_inverse(M)
    qddot_neumann = neumann_inv @ b
    torch.cuda.synchronize()
    t1 = time.time()
    results['neumann_time'].append((t1 - t0) * 1000)
    results['neumann_error'].append((torch.linalg.norm(qddot_neumann - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    qddot_neumann_pert = neumann_series_inverse(M_pert) @ b
    kappa_neumann = (torch.linalg.norm(qddot_neumann_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                    (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['neumann_kappa'].append(kappa_neumann.item())

    # --- SPAI ---
    torch.cuda.synchronize()
    t0 = time.time()
    spai_inv = spai_inverse(M)
    qddot_spai = spai_inv @ b
    torch.cuda.synchronize()
    t1 = time.time()
    results['spai_time'].append((t1 - t0) * 1000)
    results['spai_error'].append((torch.linalg.norm(qddot_spai - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    spai_inv_pert = spai_inverse(M_pert)
    qddot_spai_pert = spai_inv_pert @ b
    kappa_spai = (torch.linalg.norm(qddot_spai_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                 (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['spai_kappa'].append(kappa_spai.item())

    # --- HALA ---
    torch.cuda.synchronize()
    t0 = time.time()
    qddot_hala = hala(M, b, num_neumann=10)
    torch.cuda.synchronize()
    t1 = time.time()
    results['hala_time'].append((t1 - t0) * 1000)
    results['hala_error'].append((torch.linalg.norm(qddot_hala - qddot_ref) / torch.linalg.norm(qddot_ref)).item())

    qddot_hala_pert = hala(M_pert, b, num_neumann=10)
    kappa_hala = (torch.linalg.norm(qddot_hala_pert - qddot_ref) / torch.linalg.norm(qddot_ref)) / \
                 (torch.linalg.norm(delta_M) / torch.linalg.norm(M))
    results['hala_kappa'].append(kappa_hala.item())

# ----------------------------
# Summary
# ----------------------------
def summarize(name, times, errors, kappas):
    print(f"{name}: Avg Time (ms): {torch.tensor(times).mean():.2f}, "
          f"Avg Error: {torch.tensor(errors).mean():.2e}, "
          f"Avg Stability (kappa): {torch.tensor(kappas).mean():.2f}")

summarize("Gauss-Jordan (ref)", results['ref_time'], [0]*num_runs, results['ref_kappa'])
summarize("Neumann Series", results['neumann_time'], results['neumann_error'], results['neumann_kappa'])
summarize("SPAI", results['spai_time'], results['spai_error'], results['spai_kappa'])
summarize("HALA", results['hala_time'], results['hala_error'], results['hala_kappa'])


Using device: cpu


AssertionError: Torch not compiled with CUDA enabled